<a href="https://colab.research.google.com/github/pevu97/MIssles-and-Rocket-Identification/blob/main/missile_rocket.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shutil
import plotly.graph_objects as go
from sklearn.metrics import confusion_matrix, classification_report

from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras import layers
from tensorflow.keras import optimizers
from tensorflow.keras.callbacks import TensorBoard


In [ ]:
!wget https://storage.googleapis.com/esmartdata-courses-files/ann-course/flying-vehicles.zip
!unzip -q flying-vehicles.zip

--2026-03-02 13:06:08--  https://storage.googleapis.com/esmartdata-courses-files/ann-course/flying-vehicles.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 173.194.217.207, 192.178.219.207, 74.125.134.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|173.194.217.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1540232238 (1.4G) [application/x-zip-compressed]
Saving to: ‘flying-vehicles.zip’

flying-vehicles.zip 100%[===================>]   1.43G  37.1MB/s    in 41s     

2026-03-02 13:06:50 (35.7 MB/s) - ‘flying-vehicles.zip’ saved [1540232238/1540232238]



In [ ]:
!rm -rf ./images

In [ ]:
base_dir = './data/planes'
raw_no_of_files = {}
classes = ['drone', 'fighter-jet', 'helicopter', 'missile', 'passenger-plane', 'rocket']
for dir in classes:
  raw_no_of_files[dir] = len(os.listdir(os.path.join(base_dir, dir)))

raw_no_of_files.items()

dict_items([('drone', 1332), ('fighter-jet', 1867), ('helicopter', 1387), ('missile', 1334), ('passenger-plane', 1424), ('rocket', 1186)])

In [ ]:
data_dir = './images'

if not os.path.exists(data_dir):
  os.mkdir(data_dir)

train_dir = os.path.join(data_dir, 'train')
valid_dir = os.path.join(data_dir, 'valid')
test_dir = os.path.join(data_dir, 'test')

for directory in (train_dir, valid_dir, test_dir):
  if not os.path.exists(directory):
    os.mkdir(directory)

train_missile_dir = os.path.join(train_dir, 'missile')
train_rocket_dir = os.path.join(train_dir, 'rocket')

valid_missile_dir = os.path.join(valid_dir, 'missile')
valid_rocket_dir = os.path.join(valid_dir, 'rocket')

test_missile_dir = os.path.join(test_dir, 'missile')
test_rocket_dir = os.path.join(test_dir, 'rocket')

dirs = [train_missile_dir, train_rocket_dir, valid_missile_dir, valid_rocket_dir, test_missile_dir, test_rocket_dir]

for dir in dirs:
  if not os.path.exists(dir):
    os.mkdir(dir)

In [ ]:
missile_fnames = os.listdir(os.path.join(base_dir, 'missile'))
rocket_fnames = os.listdir(os.path.join(base_dir, 'rocket'))


In [ ]:
missile_fnames = [fname for fname in missile_fnames if fname.split('.')[1].lower() in ['jpg', 'png', 'jpeg']]
rocket_fnames = [fname for fname in rocket_fnames if fname.split('.')[1].lower() in ['jpg', 'png', 'jpeg']]


In [ ]:
size = min(len(missile_fnames), len(rocket_fnames))
size

1173

In [ ]:
train_size = int(np.floor(0.7 * size))
valid_size = int(np.floor(0.2 * size))
test_size = size - train_size - valid_size

train_size, valid_size, test_size

train_idx = train_size
valid_idx = train_size + valid_size
test_idx = train_size + valid_size + test_size

In [ ]:

for i, fname in enumerate(missile_fnames):
    if i <= train_idx:
        src = os.path.join(base_dir, 'missile', fname)
        dst = os.path.join(train_missile_dir, fname)
        shutil.copyfile(src, dst)
    elif train_idx < i <= valid_idx:
        src = os.path.join(base_dir, 'missile', fname)
        dst = os.path.join(valid_missile_dir, fname)
        shutil.copyfile(src, dst)
    elif valid_idx < i < test_idx:
        src = os.path.join(base_dir, 'missile', fname)
        dst = os.path.join(test_missile_dir, fname)
        shutil.copyfile(src, dst)

for i, fname in enumerate(rocket_fnames):
    if i <= train_idx:
        src = os.path.join(base_dir, 'rocket', fname)
        dst = os.path.join(train_rocket_dir, fname)
        shutil.copyfile(src, dst)
    elif train_idx < i <= valid_idx:
        src = os.path.join(base_dir, 'rocket', fname)
        dst = os.path.join(valid_rocket_dir, fname)
        shutil.copyfile(src, dst)
    elif valid_idx < i < test_idx:
        src = os.path.join(base_dir, 'rocket', fname)
        dst = os.path.join(test_rocket_dir, fname)
        shutil.copyfile(src, dst)

print('missile - zbiór treningowy', len(os.listdir(train_missile_dir)))
print('missile - zbiór walidacyjny', len(os.listdir(valid_missile_dir)))
print('missile - zbiór testowy', len(os.listdir(test_missile_dir)))

print('rocket - zbiór treningowy', len(os.listdir(train_rocket_dir)))
print('rocket - zbiór testowy', len(os.listdir(test_rocket_dir)))

missile - zbiór treningowy 822
missile - zbiór walidacyjny 234
missile - zbiór testowy 117
rocket - zbiór treningowy 822
rocket - zbiór testowy 117
